# Darcy face-jump indicator

Equations (5.1)–(5.3) of Araya et al. (2013), [DOI 10.1137/120888223](https://doi.org/10.1137/120888223), measure broken pressure jumps. Exact local lifts are assumptions of the reliability theorem; this notebook uses finite P2 local spaces and does not infer a certified bound from the jump indicator alone.


The current main equations and their physical checks execute from source. The full acquisition and its field galleries run with `--study`; inspect `--study --plan` for locked commands and resource requirements. Exact original external comparisons require `--historical` and their attributed payloads.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/40f4ac03d8e58e8b201498af16fadfb51fe2cac6f477a0be68e798bc53a56823/42_darcy_jump_estimator-companion.zip"
COMPANION_SHA256 = "40f4ac03d8e58e8b201498af16fadfb51fe2cac6f477a0be68e798bc53a56823"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/42_darcy_jump_estimator.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


In [ ]:
import json
from pathlib import Path
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits
from pymhm import TriangleMesh
from pymhm.estimators.darcy_jump import estimate_darcy_jumps


## Declare the local and global equations

The application writes local A, B and C blocks with the shared FEM kernels,
then supplies them to the generic assembler. Primal coefficients satisfy

$$
\begin{aligned}
a_T(p,v)&=(K\nabla p,\nabla v)_T,\\
L_T(v)&=(f,v)_T+\sum_j\alpha_jv(x_j),\\
B_T\lambda&=\langle\lambda,v\rangle_{\partial T},\\
C_Tp&=-\langle p,\mu\rangle_{\partial T}.
\end{aligned}
$$

Signs follow the fixed macroface normals. RT0 instead uses flux, pressure and
private boundary-pressure coefficients, including the local normal-flux
constraint. The global `Equation` supplies the signed Dirichlet moments and
retains each constant pressure mode. Pure Neumann data add the physical pressure
integral through an explicit constraint. `define_darcy` returns equations;
`assemble`, the global solve and field interpretation are separate stages.


In [ ]:
from pymhm import assemble, SolverConfig
from examples.formulations.darcy import define_darcy, pressure_constraints, recover_darcy

mesh = TriangleMesh.unit_square(2)
natural = {int(face): 0.0 for face in mesh.boundary_faces}
with threadpool_limits(1):
    solution_definition = define_darcy(
        mesh,
        degree=2,
        local_refinement=2,
        neumann=natural,
        source=lambda x: 8 * np.pi**2 * np.prod(np.cos(2 * np.pi * x), axis=1),
        quadrature_order=10,
    )
    solution_system = assemble(solution_definition.problem)
    solution = recover_darcy(
        solution_definition,
        solution_system,
        solution_system.solve(
            constraints=pressure_constraints(solution_definition, solution_system)
        ),
    )
    indicator = estimate_darcy_jumps(solution, neumann=natural, calibration=3.0)
print({"jump_indicator": indicator.total, "macro_cells": len(mesh.cells)})


## Five-level numerical record

The archived sequences use n=4,8,16,32,64, four fine triangles per macro, and local/trace degree pairs (2,0) and (5,3). The paper's calibrations 3 and 50 are retained. The norm is a sum of a macro-diameter-weighted broken flux H(div) term, H-inverse pressure L2 error and broken gradient error. The constant-trace indicator lies below this finite-local error; the cubic-trace ratio tends to 1.684.


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    rows = json.loads((root / "examples/results/darcy-jump/campaign.json").read_text())
    for row in rows:
        print(row["trace_degree"], row["resolution"], row["paper_error_sum"], row["indicator"], row["effectivity"])
    display(Image(filename=str(root / "docs/figures/darcy-jump/refinement.png")))
